## Transformers: BERTopic para el discurso experto de think tanks chilenos (2019–2024)


- **Programa:** Diploma de Postítulo en Inteligencia Artificial
- **Institución:** Facultad de Ciencias Físicas y Matemáticas, Universidad de Chile
- **Curso:** Procesamiento de Lenguaje Natural

Profesor Auxiliar: Rodrigo Molina Ávila - Sociólogo - MSc. Data Science - IMFD, IDIA

- **Contacto:** [Perfil IDIA](https://idia.uchile.cl/quienes-somos/rodrigo-molina/) · [GitHub](https://github.com/RodrigoMolinaAvila) · [ORCID](https://orcid.org/0009-0006-0648-0799)
- **Fecha:** 05 de Noviembre de 2026


### Resumen

Este notebook presenta un caso aplicado de **modelado de tópicos neuronales** con BERTopic, Grootendorst, M. (2022) sobre un corpus de **11.924 documentos** publicados por think tanks chilenos entre **2019 y 2024**. El corpus es una versión acotada y ampliada de la base construida para el artículo de González Hernando (2026) sobre la infraestructura intelectual chilena. El material recorre el pipeline completo: embeddings, UMAP, HDBSCAN y c-TF-IDF, discute cada decisión de diseño y propone validaciones y visualizaciones interactivas para conectar los tópicos con las variables organizacionales del corpus.

El objetivo es formativo: que el estudiante comprenda **qué hace cada componente**, **por qué** se usa y **cómo interpretar** sus resultados, incluidas sus limitaciones. Los hallazgos son exploratorios y no miden influencia política.



### Requisitos de ejecución

- Python 3.10 o superior y las dependencias de `requirements.txt`.
- CPU es suficiente; la primera ejecución descarga el modelo de embeddings y calcula la matriz (varios minutos).
- Los embeddings se guardan en `data/think_tanks_subset_embeddings_384_v2.npy`; las ejecuciones posteriores los reutilizan.
- El corpus debe estar en `data/think_tanks_subset.parquet`.

### Referencias

- Campello, R. J. G. B., Moulavi, D., & Sander, J. (2013). Density-Based Clustering Based on Hierarchical Density Estimates. *PAKDD 2013*.
- Doogan, C., & Buntine, W. (2022). Topic Model Evaluation in the Era of Large Language Models. arXiv:2209.10222.
- González Hernando, M. (2026). Think Tanks and Political Crises in Chile, 2011–2022; or, The Crystallization of the Intellectual Infrastructure in a Middle-Income Country. *Latin American Research Review*. https://doi.org/10.1017/lar.2026.10142
- Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure. arXiv:2203.05794.
- McInnes, L., Healy, J., & Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction. arXiv:1802.03426.
- Medvetz, T. (2012). *Think Tanks in America*. University of Chicago Press.
- Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. *EMNLP-IJCNLP 2019*.
- Röder, M., Both, A., & Hinneburg, A. (2015). Exploring the Space of Topic Coherence Measures. *WSDM 2015*.


In [112]:
import os
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import optuna
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer
from transformers import pipeline
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

SEED = 42  # semilla global para UMAP, Optuna y muestreos


## 0. Contexto: think tanks, crisis política y discurso experto

Los *think tanks* son organizaciones híbridas que producen y circulan conocimiento experto para intervenir en el debate público. Medvetz (2012) los describe como actores situados en la intersección de cuatro campos, academia, política, medios y financiamiento, de los que obtienen recursos y legitimidad sin pertenecer plenamente a ninguno. Esa hibridez los vuelve sensibles a los cambios de su entorno: cuando el sistema político se reordena, también se reordena lo que publican y cómo lo publican.

Chile entre 2019 y 2024 ofrece un caso privilegiado para observar esa sensibilidad. El estallido de octubre de 2019, el plebiscito de 2020, los dos procesos constituyentes y los cambios de gobierno configuraron un periodo de alta incertidumbre institucional. En ese contexto, los think tanks debieron sostener su presencia pública y, al mismo tiempo, reinterpretar temas y posiciones.

Este notebook trabaja con una **versión acotada y ampliada** de la base construida para el artículo de González Hernando (2026), *Think Tanks and Political Crises in Chile, 2011–2022*, publicado en *Latin American Research Review* (https://doi.org/10.1017/lar.2026.10142). El artículo original analiza 13.613 intervenciones de 24 organizaciones entre 2019 y 2023; la versión usada aquí conserva 11.924 documentos publicados entre 2019 y 2024.

La unidad de análisis es la **intervención**: un acto de habla con contenido cognitivo que se hace público para intervenir en la conversación política (Eyal & Buchholz, 2010). Incluye columnas, reportes, libros y seminarios. El corpus captura la *huella pública archivada* de las organizaciones: lo que decidieron publicar y mantener disponible, no la totalidad de su actividad. Por eso sobrerrepresenta productos mediáticos y refleja desigualdades de capacidad de producción. Leer los resultados sin ese contexto lleva a confundir visibilidad con influencia.

El modelado de tópicos permite describir, a escala, qué temas estructuran ese discurso y cómo se distribuyen entre organizaciones y en el tiempo. BERTopic es adecuado porque combina representaciones semánticas contextuales con clustering basado en densidad y una representación de tópicos interpretable.


## 1. Ingesta de base de datos

Cargamos la **versión acotada** del corpus de think tanks chilenos.

### Diccionario de variables

| Variable | Tipo | Descripción |
|---|---|---|
| `ID` | texto | Identificador del documento |
| `Think_Tank` | categórica | Organización que publica la intervención |
| `Tipo_Think_Tank` | categórica | Transversal, de partido o semi difunto |
| `Orientacion_Politica` | categórica | Derecha o izquierda |
| `Autor` | texto | Autoría declarada, si existe |
| `Titulo` | texto | Título del documento |
| `FechaPublicacion` | fecha | Fecha de publicación |
| `Año`, `Mes`, `Dia` | numérica | Componentes de la fecha |
| `Medio` | categórica | Medio de publicación, si aplica |
| `Producto` | categórica | Tipo de producto editorial |
| `Corpus` | texto | Texto completo de la intervención |
| `texto` | texto | Texto usado para el modelado |

La columna `texto` es el insumo del pipeline. Las variables categóricas permiten segmentar los resultados y son la base del análisis exploratorio de la sección 5.


In [113]:
import pandas as pd
df = pd.read_parquet('data/think_tanks_subset.parquet')
print('Documentos:', len(df))
print(df['Think_Tank'].value_counts().head())
print('Tipos:', df['Tipo_Think_Tank'].value_counts())


Documentos: 11924
Think_Tank
LyD                       2649
FPP                       1942
CEP                       1863
Espacio Público           1415
Fundación Jaime Guzmán     744
Name: count, dtype: int64
Tipos: Tipo_Think_Tank
Transversal      10490
De partido        1226
Semi Difuntos      208
Name: count, dtype: int64


In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11924 entries, 0 to 11923
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ID                    11924 non-null  int64         
 1   Think_Tank            11924 non-null  object        
 2   Tipo_Think_Tank       11924 non-null  object        
 3   Orientacion_Politica  11924 non-null  object        
 4   Autor                 5033 non-null   object        
 5   Titulo                11923 non-null  object        
 6   FechaPublicacion      11924 non-null  datetime64[ns]
 7   Año                   11924 non-null  float64       
 8   Mes                   11924 non-null  float64       
 9   Dia                   11924 non-null  float64       
 10  fecha_source          11924 non-null  object        
 11  fecha_aprox           11924 non-null  bool          
 12  Medio                 2338 non-null   object        
 13  Corpus          

### 1.1 Composición del corpus

Antes de modelar conviene describir el corpus con las variables disponibles. El panel muestra cuatro cortes: concentración por organización, tipo de think tank, orientación política y evolución anual. `df.info()` permite verificar tipos, valores nulos y tamaño en memoria; los conteos permiten detectar desbalances que después condicionan los tópicos (por ejemplo, una organización que produce diez veces más que otra).

La concentración no es un defecto del corpus: es parte del fenómeno. Si un think tank produce más documentos, tendrá más peso en el vocabulario y en los tópicos. Por eso el análisis de la sección 5 normaliza por organización cuando compara agendas.

In [115]:
conteos = {
    'Think tank (top 12)': df['Think_Tank'].value_counts().head(12).sort_values(),
    'Tipo de think tank': df['Tipo_Think_Tank'].value_counts().sort_values(),
    'Orientación política': df['Orientacion_Politica'].value_counts().sort_values(),
    'Año de publicación': df['Año'].astype(int).value_counts().sort_index(),
}

fig = make_subplots(rows=2, cols=2, subplot_titles=list(conteos))
posiciones = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (titulo, serie), (fila, col) in zip(conteos.items(), posiciones):
    fig.add_trace(
        go.Bar(x=serie.values, y=serie.index.astype(str), orientation='h',
               hovertemplate='%{y}: %{x} documentos<extra></extra>',
               showlegend=False),
        row=fila, col=col,
    )

fig.update_layout(
    height=720, template='plotly_white',
    title='Composición del corpus (11.924 documentos, 2019–2024)',
)
fig.show()


## 2. Preprocesamiento y unidad de texto

El preprocesamiento cumple dos funciones distintas: preparar el texto para el **vectorizador de tópicos** (c-TF-IDF) y prepararlo para el **modelo de embeddings**. No conviene aplicar las mismas decisiones a ambos.

### 2.1 Normalización

- Minúsculas: reduce la duplicación de formas.
- Eliminación de URLs y correos: no aportan contenido temático e introducen ruido.
- Filtrado de caracteres: conserva letras españolas y espacios; elimina puntuación, números y símbolos.
- Colapso de espacios.

No se aplica stemming ni lematización: SBERT representa palabras en contexto y el stemming destruiría información morfológica; para c-TF-IDF se usan n-gramas (1,2) que capturan colocaciones.

### 2.2 Stopwords

Se eliminan palabras funcionales y vocabulario académico genérico. La lista es deliberadamente agresiva y combina inglés y español porque el corpus contiene documentos en ambas lenguas. Se excluyen además términos genéricos del discurso académico ("results", "analysis", "estudio") que dominarían las listas de palabras sin distinguir tópicos.

Riesgo: la lista también elimina términos con contenido ("modelo", "tiempo", "parte"), lo que puede borrar matices. En un análisis definitivo conviene tratar la lista como un hiperparámetro y evaluar su efecto.




In [ ]:
import re

academic_stops = [
    "study","results","analysis","data","method","using","used","based",
    "research","paper","different","new","proposed","however","also",
    "doi","copyright","rights","reserved","elsevier","springer","ieee",
    "et","al","table","figure","significant","observed","showed",
    "author","introduction","conclusion","discussion",
    "estudio","análisis","datos","método","resultados","investigación",
    "durante","mediante","través","además","parte","embargo","resumen",
    "article",
]
stops_en = ["the","and","of","to","in","a","is","for","that","on","with","as","by",
            "this","are","it","from","at","or","an","be","which","can"]
extra_stops_en = [
    "was","were","have","has","had","having","been","not","these","those","their","our","its",
    "such","between","over","under","through","during","more","most","other","others",
    "any","all","than","when","where","while","about","into","after","before","each",
    "both","same","some","then","there","here","whose","whom","we","us","you","your",
    "they","them","he","she","his","her","him","do","does","did","done","will","would",
    "could","should","may","might","must","shall","if","so","but","because","thus",
    "therefore","hence","either","neither","only","just","still","yet","even","much",
    "many","few","several","various","per","via","upon","within","without","among",
    "across","throughout","toward","towards","whether","one","two","three","first",
    "second","third","present","found","show","shows","showing","shown",
    "approach","level","levels","group","groups","case","cases",
    "how","why","what","who",
    "high","associated","model","models","use","role","potential","increased",
    "compared","process","performance","conditions","time",
]
stops_es = ["de","la","que","el","en","y","a","los","del","se","las","por","un",
            "para","con","no","una","su","al","lo","como","más","pero","sus"]
extra_stops_es = [
    "entre","sobre","cada","otro","otra","otros","otras","tiene","tienen","fue","fueron",
    "son","era","eran","haber","hay","muy","mas","sin","este","esta","estos","estas",
    "ese","esa","esos","esas","cual","cuales","cuando","donde","mientras","hacia",
    "desde","hasta","segun","según","ademas","además","tras","sino","ni","tanto",
]
chile_stops = ["chile","chilean","chilena","chileno"]

STOPWORDS = set(academic_stops + stops_en + extra_stops_en + stops_es + extra_stops_es + chile_stops)
print(f"Total stopwords base: {len(STOPWORDS)}")

BOILERPLATE_PATTERNS = [
    re.compile(r'las opiniones expresadas en esta publicaci[oó]n.{0,400}?u otros miembros\.?', re.I | re.S),
    re.compile(r'columna de [^.]{0,180}?publicada en [^.]{0,100}?\.-?', re.I),
    re.compile(r'(?:recibe toda la informaci[oó]n de ir ideas republicanas|todos los derechos reservados).{0,200}?mrdesign[^.]{0,80}', re.I | re.S),
    re.compile(r'(?:revisa|revisar|descarga|descargar|lee|leer)[^.]{0,100}?aqu[ií]?[^.]{0,40}\.?', re.I),
    re.compile(r's[ií]guenos en nuestras redes sociales.{0,200}?(?:fundamental|apoyo)\.?', re.I | re.S),
    re.compile(r'desarrollado por integratek[^.]{0,120}', re.I),
]


def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto)
    texto = re.sub(r'\S+@\S+', '', texto)
    for patron in BOILERPLATE_PATTERNS:
        texto = patron.sub(' ', texto)
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['texto_limpio'] = df['texto'].apply(limpiar_texto)
print('Ejemplo de texto limpio:')
print(df['texto_limpio'].iloc[0][:300])


Total stopwords base: 269
Ejemplo de texto limpio:
continua la desaceleración en la creación de empleo el instituto nacional de estadísticas ine dio a conocer las cifras laborales correspondientes al trimestre móvil septiembre noviembre de la tasa de desocupación nacional fue de puntos porcentuales pp por debajo del mismo período del año anterior es


### 2.2 bis Plantillas de boilerplate

Las stopwords no bastan cuando el ruido aparece como **frase completa**. Antes de vectorizar se eliminan plantillas verificadas: el descargo de FPP (16,2 % del corpus), la firma de LyD (`Columna de… publicada en…`, 9,2 %), el footer de IR (`mrdesign`, 0,4 %), los CTA `revisa/descarga… aquí` (2,6 %) y el crédito de Integratek (1,4 %). Los patrones tienen límites acotados para no borrar contenido.

Este paso ocurre antes de calcular embeddings; por eso el cache cambia a `think_tanks_subset_embeddings_384_v2.npy`. Si ejecutas el notebook con el cache anterior, los modelos no reflejarán la limpieza.

### 2.3 Chunking y límite de tokens

Un documento puede superar el máximo de tokens que el modelo procesa. `max_seq_length` se expresa en **tokens de subpalabra**, no en palabras: en este corpus limpio, 180 palabras equivalen a una mediana de ~224 tokens y un percentil 95 de ~248. Con `max_seq_length=256` se truncaba cerca del 1,8% de los chunks; con 384 no se trunca ninguno (máximo observado: 378).

El chunking divide el texto en fragmentos de 180 palabras con un solape de 30 (paso de 150) para no perder contexto en los bordes. El embedding del documento es el promedio de los embeddings de sus chunks, normalizado en L2:

$$\mathbf{e}_d = \frac{1}{|C_d|}\sum_{c \in C_d}\mathbf{e}_c, \qquad \hat{\mathbf{e}}_d = \frac{\mathbf{e}_d}{\lVert \mathbf{e}_d \rVert_2}$$

El procedimiento no es inocuo: mezcla secciones del documento y diluye documentos muy largos. Es un compromiso entre cobertura y fidelidad.


In [117]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
EMB_PATH = 'data/think_tanks_subset_embeddings_384_v2.npy'

def embed_con_chunks(model, texts, max_seq=384, chunk_words=180, overlap_words=30, batch=64):
    model.max_seq_length = max_seq
    chunks, owner = [], []
    for i, t in enumerate(tqdm(texts, desc='Chunking')):
        words = t.split()
        if len(words) <= chunk_words + overlap_words:
            chunks.append(t); owner.append(i)
        else:
            step = chunk_words - overlap_words
            for s in range(0, len(words), step):
                part = words[s:s+chunk_words]
                if len(part) < 20 and chunks: break
                chunks.append(' '.join(part)); owner.append(i)
    print(f'{len(chunks)} chunks de {len(texts)} docs')
    emb_chunks = model.encode(chunks, batch_size=batch, show_progress_bar=True, convert_to_numpy=True)
    emb = np.zeros((len(texts), emb_chunks.shape[1]), dtype=np.float32)
    counts = np.zeros(len(texts))
    for j, i in enumerate(tqdm(owner, desc='Agregando chunks')):
        emb[i] += emb_chunks[j]; counts[i] += 1
    emb /= np.maximum(counts, 1)[:, None]
    emb /= (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
    return emb

if os.path.exists(EMB_PATH):
    embeddings = np.load(EMB_PATH)
    print('Embeddings cargados:', embeddings.shape)
else:
    print('Descargando modelo y calculando embeddings...')
    model = SentenceTransformer(MODEL_NAME)
    embeddings = embed_con_chunks(model, df['texto_limpio'].tolist())
    np.save(EMB_PATH, embeddings)
    print('Embeddings guardados:', embeddings.shape)


Descargando modelo y calculando embeddings...


Chunking:   0%|          | 0/11924 [00:00<?, ?it/s]

49793 chunks de 11924 docs


Batches:   0%|          | 0/779 [00:00<?, ?it/s]

Agregando chunks:   0%|          | 0/49793 [00:00<?, ?it/s]

Embeddings guardados: (11924, 384)


### 3.1 Embeddings: Sentence-BERT (Grootendorst, 2022, §3.1)

BERTopic convierte cada documento en un vector denso con un modelo de *sentence embeddings*. Usamos **Sentence-BERT** (SBERT; Reimers & Gurevych, 2019), una arquitectura tipo *bi-encoder*: dos textos se codifican de forma independiente y la similitud se calcula con el producto punto o la similitud coseno. Esto es mucho más eficiente que un *cross-encoder*, que requeriría evaluar cada par de documentos.

El modelo `paraphrase-multilingual-MiniLM-L12-v2` es un transformer multilingüe de 12 capas que produce vectores de **384 dimensiones**. A diferencia de un embedding estático (word2vec, GloVe), el vector de una palabra depende de su contexto; por eso SBERT no necesita eliminar stopwords para representar significado.

El *pooling* es la operación que convierte la secuencia de estados ocultos en un único vector. Este modelo usa **mean pooling**: promedia los token embeddings. Luego normalizamos cada vector en L2 para que la similitud coseno dependa solo del ángulo.

Para documentos largos aplicamos **chunking**: dividimos el texto en fragmentos de 180 palabras con solape de 30, calculamos el embedding de cada fragmento y promediamos. El límite `max_seq_length=384` tokens evita el truncamiento silencioso: con 256 se perdía el final de cerca del 1,8% de los chunks. El resultado es una matriz de 11.924 × 384.


In [118]:
from sklearn.decomposition import PCA

rng = np.random.default_rng(SEED)
idx = rng.choice(len(embeddings), size=min(3000, len(embeddings)), replace=False)

pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(embeddings[idx])

proj = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'Think tank': df['Think_Tank'].iloc[idx].values,
    'Orientación': df['Orientacion_Politica'].iloc[idx].values,
    'Año': df['Año'].iloc[idx].astype('Int64').astype(str).values,
    'Título': df['Titulo'].iloc[idx].fillna('').str.slice(0, 90).values,
})

fig = px.scatter(
    proj, x='PC1', y='PC2', color='Orientación',
    hover_data=['Think tank', 'Año', 'Título'],
    opacity=0.55, render_mode='webgl',
    title=f'PCA 2D de una muestra de embeddings '
          f'(varianza explicada: {pca.explained_variance_ratio_.sum():.1%})',
)
fig.update_traces(marker=dict(size=4))
fig.show()


### 3.2 UMAP (Grootendorst, 2022, §3.2)

**UMAP** (*Uniform Manifold Approximation and Projection*; McInnes et al., 2018) es un método de reducción de dimensionalidad no lineal. Construye una representación topológica difusa de los datos en alta dimensión (un complejo simplicial) y busca una proyección de baja dimensión que preserve esa estructura. Frente a PCA, captura relaciones no lineales; frente a t-SNE, conserva mejor la estructura global y es más rápido.

Parámetros relevantes:

- `n_neighbors=15`: equilibrio entre estructura local y global. Valores bajos enfatizan vecindarios muy locales.
- `n_components=5`: dimensión de salida para el clustering. No usamos 2D para HDBSCAN porque perderíamos información.
- `min_dist=0.0`: permite que los puntos se agrupen densamente; adecuado para clustering.
- `metric='cosine'`: coherente con embeddings normalizados.
- `random_state=SEED`: sin semilla, UMAP no es reproducible.

Usamos **tres proyecciones**: una de 5 dimensiones como entrada de HDBSCAN, y proyecciones de 2 y 3 dimensiones solo para visualizar. La versión 3D permite rotar la nube y separar visualmente grupos que en 2D quedan superpuestos. Las distancias en UMAP no son métricas: no deben interpretarse como distancias semánticas absolutas.


In [119]:
from umap import UMAP

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric='cosine', random_state=SEED)
umap_embeddings = umap_model.fit_transform(embeddings)
print('UMAP (5D) output shape:', umap_embeddings.shape)

# Proyección de 2 dimensiones, solo para visualización
umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.0,
               metric='cosine', random_state=SEED)
emb_2d = umap_2d.fit_transform(embeddings)

# Proyección de 3 dimensiones, solo para exploración interactiva
umap_3d = UMAP(n_neighbors=15, n_components=3, min_dist=0.0,
               metric='cosine', random_state=SEED)
emb_3d = umap_3d.fit_transform(embeddings)

viz = pd.DataFrame({
    'UMAP 1': emb_2d[:, 0],
    'UMAP 2': emb_2d[:, 1],
    'UMAP 3': emb_3d[:, 2],
    'Think tank': df['Think_Tank'].values,
    'Orientación': df['Orientacion_Politica'].values,
    'Año': df['Año'].astype('Int64').astype(str).values,
    'Título': df['Titulo'].fillna('').str.slice(0, 90).values,
})

fig = px.scatter(
    viz, x='UMAP 1', y='UMAP 2', color='Orientación',
    hover_data=['Think tank', 'Año', 'Título'],
    opacity=0.55, render_mode='webgl',
    title='Proyección UMAP 2D de los embeddings (color: orientación política)',
)
fig.update_traces(marker=dict(size=4))
fig.show()

fig3d = px.scatter_3d(
    viz, x='UMAP 1', y='UMAP 2', z='UMAP 3', color='Orientación',
    hover_data=['Think tank', 'Año', 'Título'],
    opacity=0.45,
    title='Proyección UMAP 3D de los embeddings (color: orientación política)',
)
fig3d.update_traces(marker=dict(size=2))
fig3d.show()


UMAP (5D) output shape: (11924, 5)


### 3.3 HDBSCAN (Grootendorst, 2022, §3.3)

**HDBSCAN** (*Hierarchical Density-Based Spatial Clustering of Applications with Noise*; Campello et al., 2013) es un algoritmo de clustering basado en densidad. A diferencia de k-means, no requiere fijar el número de clusters y puede descubrir grupos de densidad y forma variables.

Su funcionamiento, en síntesis:

1. Calcula la **distancia de alcanzabilidad mutua** entre puntos, que combina la distancia euclidiana con la densidad local (distancia al k-ésimo vecino).
2. Construye un **árbol de expansión mínima** sobre ese grafo.
3. Deriva una **jerarquía de clusters** y la poda con el criterio *excess of mass* (EOM), que selecciona los clusters más estables.

Parámetros:

- `min_cluster_size=30`: tamaño mínimo de un grupo. Es el parámetro más influyente.
- `min_samples=10`: controla la robustez frente a ruido; valores mayores producen más outliers.
- `prediction_data=True`: habilita la predicción aproximada para nuevos documentos y es necesario para algunas operaciones de BERTopic.

Los documentos que no pertenecen a ningún cluster reciben la etiqueta `-1` (**outliers**). No son errores: son señal de que el espacio no es denso en esa región. Su proporción debe reportarse siempre.

La figura 2D resume la partición; la figura 3D permite rotar la nube y comprobar si los clusters se separan en el espacio o solo se ven contiguos por la proyección. Ambas usan el mismo etiquetado y los mismos colores.


In [120]:
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(min_cluster_size=30, min_samples=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
clusters = hdbscan_model.fit_predict(umap_embeddings)
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_outliers = list(clusters).count(-1)
print(f'Clusters: {n_clusters}, Outliers: {n_outliers} ({n_outliers/len(clusters):.1%})')


Clusters: 62, Outliers: 5208 (43.7%)


In [121]:
viz['Cluster'] = [f'Tópico {c}' if c != -1 else 'Outlier (-1)' for c in clusters]
grandes = [f'Tópico {c}' for c in pd.Series(clusters).value_counts().index
           if c != -1][:9]
viz['Grupo'] = viz['Cluster'].where(viz['Cluster'].isin(grandes),
                                   'Otros clusters / outliers')

fig = px.scatter(
    viz, x='UMAP 1', y='UMAP 2', color='Grupo',
    hover_data=['Think tank', 'Año', 'Título'],
    opacity=0.55, render_mode='webgl',
    title='Clusters HDBSCAN sobre la proyección UMAP 2D',
)
fig.update_traces(marker=dict(size=4))
fig.show()

fig3d = px.scatter_3d(
    viz, x='UMAP 1', y='UMAP 2', z='UMAP 3', color='Grupo',
    hover_data=['Think tank', 'Año', 'Título'],
    opacity=0.45,
    title='Clusters HDBSCAN sobre la proyección UMAP 3D',
)
fig3d.update_traces(marker=dict(size=2))
fig3d.show()


### 3.4 c-TF-IDF (Grootendorst, 2022, §3.4)

**c-TF-IDF** (*class-based TF-IDF*) es la contribución central de BERTopic. En lugar de calcular TF-IDF por documento, agrupa todos los documentos de un tópico en un único "documento" y calcula la frecuencia de términos dentro de ese grupo. La fórmula es:

$$W_{t,c} = tf_{t,c} \cdot \log\left(1 + \frac{A}{f_t}\right)$$

donde $tf_{t,c}$ es la frecuencia del término $t$ en la clase $c$, $f_t$ es la frecuencia del término en todo el corpus y $A$ es el número promedio de palabras por clase. Un término recibe un peso alto si es frecuente dentro de su tópico y raro en los demás: eso lo hace **discriminativo**.

El `CountVectorizer` define el vocabulario con n-gramas de 1 y 2 palabras, `min_df=2` y la lista de stopwords. `ClassTfidfTransformer` aplica la transformación y normaliza. Aquí sí importan las stopwords, aunque no fueran necesarias para SBERT: sin eliminarlas, dominarían las listas de todos los tópicos.


In [122]:
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

# c-TF-IDF: representación de tópicos
# Primero, agrupar documentos por cluster
docs_by_cluster = {}
for i, c in enumerate(clusters):
    docs_by_cluster.setdefault(c, []).append(df['texto_limpio'].iloc[i])

# Vectorizador
vectorizer = CountVectorizer(stop_words=list(STOPWORDS), ngram_range=(1,2), min_df=2)
X = vectorizer.fit_transform([' '.join(docs) for docs in docs_by_cluster.values()])
ctfidf = ClassTfidfTransformer()
c_tfidf_matrix = ctfidf.fit_transform(X)
feature_names = vectorizer.get_feature_names_out()

# Mostrar top palabras por tópico
for idx, cluster in enumerate(docs_by_cluster.keys()):
    if cluster == -1: continue
    scores = c_tfidf_matrix[idx].toarray().flatten()
    top_idx = scores.argsort()[-10:][::-1]
    words = [feature_names[i] for i in top_idx]
    print(f'Tópico {cluster}: {", ".join(words)}')


Tópico 29: empleo, trimestre, tasa, desempleo, meses, ocupados, trabajo, desocupación, fuerza trabajo, empleos
Tópico 56: artículo, comisión, indicación, convencional, convención, constituyente, votación, estado, boletín informativo, informativo
Tópico 10: pensiones, sistema, pensión, afp, reforma, cotización, fondos, previsional, es, años
Tópico 3: migrantes, migración, inmigrantes, migrante, población, inmigración, extranjeros, migratoria, país, población migrante
Tópico 4: salud, isapres, fonasa, sistema, medicamentos, espera, sistema salud, afiliados, ley, precios
Tópico 34: transporte, ley, tarjetas, transporte público, pago, empresas, buses, proyecto, clientes, servicio
Tópico 50: partidos, electoral, sistema, electorales, sistema electoral, elecciones, fragmentación, partidos políticos, políticos, representación
Tópico 32: horas, laboral, trabajadores, jornada, trabajo, empresas, jornada laboral, es, empleo, productividad
Tópico 2: educación, estudiantes, escolar, establecimient

In [123]:
top_clusters = [c for c in pd.Series(clusters).value_counts().index
               if c != -1][:6]

fig = make_subplots(rows=2, cols=3,
                    subplot_titles=[f'Tópico {c}' for c in top_clusters])

for k, c in enumerate(top_clusters):
    fila = list(docs_by_cluster).index(c)
    scores = c_tfidf_matrix[fila].toarray().flatten()
    top_idx = scores.argsort()[-8:][::-1]
    fig.add_trace(
        go.Bar(x=scores[top_idx],
               y=[feature_names[i] for i in top_idx],
               orientation='h', showlegend=False,
               hovertemplate='%{y}: %{x:.3f}<extra></extra>'),
        row=1 + k // 3, col=1 + k % 3,
    )

fig.update_layout(height=650, template='plotly_white',
                  title='c-TF-IDF: términos más discriminativos por cluster (top 6)')
fig.show()


### 3.5 Integración en BERTopic

BERTopic es un **orquestador modular**: recibe los componentes ya construidos y ejecuta la secuencia completa. Al pasar `umap_model`, `hdbscan_model` y `vectorizer_model`, controlamos cada etapa y podemos reproducir el experimento.

- `fit_transform(docs, embeddings)` ajusta el modelo y devuelve `topics` y `probs`.
- `topics` contiene la etiqueta de cluster de cada documento; `-1` es outlier.
- Los identificadores de tópico **no están ordenados por tamaño**; siempre hay que consultar `get_topic_info()`.
- `get_topic(t)` devuelve las palabras y pesos c-TF-IDF del tópico `t`.
- `get_representative_docs(t)` devuelve documentos cercanos al centroide del tópico.

`verbose=True` imprime el progreso de cada etapa. En producción conviene además fijar semillas y guardar el modelo con `save()`.


In [124]:
from bertopic import BERTopic

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    verbose=True
)
topics, probs = topic_model.fit_transform(df['texto_limpio'].tolist(), embeddings)
print(topic_model.get_topic_info().head(10))


2026-11-05 18:26:14,368 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-05 18:26:21,265 - BERTopic - Dimensionality - Completed ✓
2026-11-05 18:26:21,267 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-05 18:26:21,686 - BERTopic - Cluster - Completed ✓
2026-11-05 18:26:21,691 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-05 18:26:34,330 - BERTopic - Representation - Completed ✓


   Topic  Count                                             Name  \
0     -1   5208                              -1_es_ms_ha_poltica   
1      0    537                                  0_es_ms_nos_ser   
2      1    510  1_educacin_estudiantes_escolar_establecimientos   
3      2    508                   2_pensiones_sistema_pensin_afp   
4      3    329   3_constitucional_constitucin_convencin_proceso   
5      4    234               4_inflacin_crecimiento_economa_pib   
6      5    207                           5_virus_es_pandemia_ms   
7      6    202                                   6_es_ms_pas_ha   
8      7    202                   7_salud_isapres_fonasa_sistema   
9      8    182                  8_partido_es_derecha_elecciones   

                                      Representation  \
0  [es, ms, ha, poltica, ser, si, gobierno, estad...   
1  [es, ms, nos, ser, le, todo, si, me, libertad,...   
2  [educacin, estudiantes, escolar, establecimien...   
3  [pensiones, sistema, pen

## 4. Validación

### 4.1 ¿Por qué validar?

El artículo de Grootendorst no prescribe un método de validación, y Doogan & Buntine (2022) advierten que las métricas automáticas de coherencia y diversidad no capturan la interpretabilidad. Un modelo de tópicos puede ser numéricamente estable y, aun así, producir etiquetas que no significan nada para un experto.

Por eso combinamos tres estrategias complementarias:

1. **Búsqueda de hiperparámetros** con Optuna y una métrica interna (silhouette).
2. **Validación semántica** con un modelo de inferencia de lenguaje natural (NLI) en modo zero-shot.
3. **Análisis de estabilidad** entre semillas con el índice ARI y comparación de vocabulario.

Ninguna de las tres reemplaza la lectura humana de los tópicos y de los documentos representativos. La validación es un filtro, no una prueba de verdad.


### 4.2 Búsqueda de hiperparámetros con Optuna

**Optuna** es una biblioteca de optimización de hiperparámetros. Su muestreador por defecto, **TPE** (*Tree-structured Parzen Estimator*), modela la probabilidad de que una configuración sea buena y concentra las evaluaciones en regiones prometedoras, en lugar de explorar aleatoriamente.

La función objetivo ajusta UMAP y HDBSCAN para cada combinación de `n_neighbors` y `min_cluster_size`, y devuelve el **coeficiente de silueta**:

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$

donde $a(i)$ es la distancia media a los puntos de su propio cluster y $b(i)$ la distancia media al cluster más cercano. El valor está en $[-1, 1]$; valores altos indican clusters compactos y bien separados.

Advertencias: la silueta supone clusters convexos y no penaliza los outliers; además, se calcula sobre el espacio UMAP, que ya deformó las distancias. Sirve para comparar configuraciones, no como medida absoluta de calidad. Con 10 *trials* hacemos una demostración, no una búsqueda exhaustiva.


In [125]:
import optuna
from sklearn.metrics import silhouette_score

def objective(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 10, 50)
    min_cluster_size = trial.suggest_int('min_cluster_size', 20, 100)
    umap_m = UMAP(n_neighbors=n_neighbors, n_components=5, min_dist=0.0, metric='cosine', random_state=SEED)
    hdb_m = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=10, metric='euclidean', prediction_data=True)
    emb = umap_m.fit_transform(embeddings)
    labels = hdb_m.fit_predict(emb)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters < 2: return -1.0
    mask = labels != -1
    if mask.sum() < 2: return -1.0
    sil = silhouette_score(emb[mask], labels[mask])
    return sil

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=10)
print('Best params:', study.best_params)
print('Best silhouette:', study.best_value)

# Refit del modelo final con los mejores hiperparámetros
best = study.best_params
topic_model = BERTopic(
    umap_model=UMAP(n_neighbors=best['n_neighbors'], n_components=5, min_dist=0.0, metric='cosine', random_state=SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=best['min_cluster_size'], min_samples=10, metric='euclidean', prediction_data=True),
    vectorizer_model=vectorizer,
    verbose=True
)
topics, probs = topic_model.fit_transform(df['texto_limpio'].tolist(), embeddings)
print('Modelo final refiteado.')


[I 2026-11-05 18:26:38,458] A new study created in memory with name: no-name-749fd338-84b9-4a3a-9531-29529044e3e6
[I 2026-11-05 18:26:48,604] Trial 0 finished with value: 0.5341554880142212 and parameters: {'n_neighbors': 25, 'min_cluster_size': 97}. Best is trial 0 with value: 0.5341554880142212.
[I 2026-11-05 18:27:01,313] Trial 1 finished with value: 0.5498054027557373 and parameters: {'n_neighbors': 40, 'min_cluster_size': 68}. Best is trial 1 with value: 0.5498054027557373.
[I 2026-11-05 18:27:09,159] Trial 2 finished with value: 0.5250155329704285 and parameters: {'n_neighbors': 16, 'min_cluster_size': 32}. Best is trial 1 with value: 0.5498054027557373.
[I 2026-11-05 18:27:16,327] Trial 3 finished with value: 0.565695583820343 and parameters: {'n_neighbors': 12, 'min_cluster_size': 90}. Best is trial 3 with value: 0.565695583820343.
[I 2026-11-05 18:27:27,747] Trial 4 finished with value: 0.5306488871574402 and parameters: {'n_neighbors': 34, 'min_cluster_size': 77}. Best is tri

Best params: {'n_neighbors': 17, 'min_cluster_size': 34}
Best silhouette: 0.5973525643348694


2026-11-05 18:28:23,953 - BERTopic - Dimensionality - Completed ✓
2026-11-05 18:28:23,955 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-05 18:28:24,338 - BERTopic - Cluster - Completed ✓
2026-11-05 18:28:24,343 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-05 18:28:36,786 - BERTopic - Representation - Completed ✓


Modelo final refiteado.


### 4.3 Validación semántica con NLI

Un modelo de **inferencia de lenguaje natural** (NLI) recibe una premisa y una hipótesis y estima si la premisa implica la hipótesis (*entailment*), la contradice (*contradiction*) o es neutral. El pipeline `zero-shot-classification` reutiliza esa capacidad para asignar etiquetas no vistas durante el entrenamiento.

En la celda, la premisa es un documento representativo del tópico y la hipótesis se construye con las palabras principales del tópico. Un puntaje alto sugiere que el documento es compatible con la etiqueta.

**Limitación importante:** la hipótesis se deriva de las mismas palabras que definen el tópico, de modo que la validación es parcialmente circular. Además, con una sola etiqueta candidata el puntaje no se compara contra alternativas. Este bloque debe leerse como una comprobación de cordura, no como una validación definitiva. Una versión más robusta usaría un conjunto fijo de etiquetas candidatas para todos los tópicos, o evaluación humana.


In [126]:
from transformers import pipeline

# Modelo NLI multilingüe (zero-shot)
nli = pipeline('zero-shot-classification', model='MoritzLaurer/mDeBERTa-v3-base-mnli-xnli')

# Validar los 3 tópicos más grandes
top_topics = topic_model.get_topic_info().head(3)['Topic'].tolist()
for t in tqdm(top_topics, desc='Validación NLI'):
    words = [w for w, _ in topic_model.get_topic(t)[:5]]
    hypothesis = f'Este documento trata sobre {", ".join(words)}.'
    # Usar el documento representativo como premisa
    rep_doc = topic_model.get_representative_docs(t)[0]
    result = nli(rep_doc[:512], candidate_labels=[hypothesis])
    print(f'Tópico {t}: {words} -> NLI score: {result["scores"][0]:.3f}')


Device set to use cuda:0


Validación NLI:   0%|          | 0/3 [00:00<?, ?it/s]

Tópico -1: ['es', 'ms', 'ha', 'poltica', 'ser'] -> NLI score: 0.605
Tópico 0: ['pensiones', 'sistema', 'pensin', 'afp', 'reforma'] -> NLI score: 0.991
Tópico 1: ['educacin', 'estudiantes', 'escolar', 'establecimientos', 'nios'] -> NLI score: 0.982


### 4.4 Estabilidad entre semillas

UMAP y HDBSCAN son estocásticos: dos ejecuciones con semillas distintas pueden producir particiones distintas. El **índice de Rand ajustado** (ARI) mide el acuerdo entre dos particiones contando pares de documentos que se agrupan juntos o separados en ambas, y corrigiendo por el acuerdo esperado al azar:

$$ARI = \frac{RI - \mathbb{E}[RI]}{\max(RI) - \mathbb{E}[RI]}$$

Su valor es 1 cuando las particiones coinciden, 0 cuando el acuerdo es el esperado por azar y negativo cuando hay menos acuerdo que por azar. Un ARI alto indica una partición estable; uno bajo, dependencia de la semilla.

Advertencia: los identificadores de tópico cambian entre ejecuciones, así que no se pueden comparar directamente. Por eso el ARI se calcula sobre las etiquetas de cluster, no sobre los nombres de tópico. El costo de repetir el pipeline tres veces es alto; en un análisis real conviene usar más semillas y reportar media y desviación.


In [127]:
from sklearn.metrics import adjusted_rand_score

def run_with_seed(seed):
    u = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=seed)
    h = HDBSCAN(min_cluster_size=30, min_samples=10, metric='euclidean', prediction_data=True)
    e = u.fit_transform(embeddings)
    l = h.fit_predict(e)
    return l

labels_1 = run_with_seed(42)
labels_2 = run_with_seed(1)
labels_3 = run_with_seed(2)
print('ARI 42 vs 1:', adjusted_rand_score(labels_1, labels_2))
print('ARI 42 vs 2:', adjusted_rand_score(labels_1, labels_3))


ARI 42 vs 1: 0.6746600216599312
ARI 42 vs 2: 0.6545487010963991


### 4.5 Top-N palabras por tópico

La etiqueta de un tópico es, en la práctica, su lista de términos c-TF-IDF. Inspeccionar un número fijo de palabras (`N_TOP_WORDS`) permite comparar tópicos, detectar etiquetas redundantes y evaluar si el vocabulario es interpretable.

Esta celda es el punto de extensión para el análisis de estabilidad léxica: puede repetirse con los clusters de cada semilla (`labels_1`, `labels_2`, `labels_3`) y compararse el solapamiento de vocabulario entre tópicos equivalentes. El ARI mide acuerdo en la asignación; el top-N mide acuerdo en la interpretación.


In [128]:
from IPython.display import display

N_TOP_WORDS = 10  # cambia este valor para inspeccionar más o menos palabras

info = topic_model.get_topic_info()
info = info[info['Topic'] != -1].head(12)

filas = []
for _, fila in info.iterrows():
    for rank, (palabra, peso) in enumerate(
            topic_model.get_topic(fila['Topic'])[:N_TOP_WORDS], start=1):
        filas.append({'Tópico': fila['Topic'], 'Rank': rank,
                      'Palabra': palabra, 'Peso c-TF-IDF': round(float(peso), 4)})

tabla_top = pd.DataFrame(filas)
tabla_ancha = tabla_top.pivot(index='Tópico', columns='Rank', values='Palabra')
tabla_ancha.columns = [f'#{i}' for i in tabla_ancha.columns]
display(tabla_ancha)

# Punto de extensión: repetir este cálculo para los clusters de cada semilla
# (labels_1, labels_2, labels_3) y comparar el solapamiento de vocabulario.


,#1,#2,#3,#4,#5,#6,#7,#8,#9,#10
Tópico,,,,,,,,,,
0,pensiones,sistema,pensin,afp,reforma,cotizacin,fondos,es,previsional,aos
1,educacin,estudiantes,escolar,establecimientos,nios,es,colegios,ms,escuelas,sistema
2,es,ms,nos,ser,todo,mundo,si,libertad,tambin,me
3,constitucional,constitucin,convencin,proceso,es,texto,propuesta,nueva,ser,nueva constitucin
4,inflacin,crecimiento,economa,banco,banco central,ao,pib,central,ms,monetaria
5,trabajadores,laboral,trabajo,horas,jornada,negociacin,empleo,es,empresas,sindicatos
6,artculo,derechos,derecho,estado,comisin,indgenas,convencin,constitucin,pueblos,constitucional
7,salud,isapres,fonasa,sistema,afiliados,espera,medicamentos,sistema salud,ley,precios
8,es,ms,pas,ha,si,estado,eso,porque,est,nos


## 5. Resultados y análisis descriptivo

### 5.1 Lectura de resultados

La tabla `get_topic_info()` es el punto de partida. Sus columnas principales son:

- `Topic`: identificador; `-1` agrupa los outliers.
- `Count`: número de documentos asignados.
- `Name`: etiqueta construida con las palabras de mayor peso.
- `Representation`: lista completa de palabras y pesos.

Buenas prácticas de lectura:

1. **Reportar la proporción de outliers** antes de interpretar cualquier tópico. Una proporción alta indica que el espacio es continuo o que los parámetros son demasiado estrictos.
2. **No ordenar por identificador**: los tópicos no están ordenados por importancia. Ordenar por `Count`.
3. **Leer documentos representativos**, no solo palabras clave. Una lista de términos puede ser coherente y, aun así, agrupar documentos heterogéneos.
4. **Nombrar los tópicos manualmente** después de inspeccionarlos. Los nombres automáticos son un apoyo, no un resultado final.
5. **Distinguir temas de géneros discursivos**: un cluster puede estar definido por un formato (entrevistas, boletines) y no por un tema.

El análisis temporal agrega una dimensión: un tópico puede crecer por un evento político, por un cambio en la estrategia editorial de una organización o por ambas cosas. Sin información externa, la interpretación causal es especulativa.


**Un modelo no alcanza.** El modelo base de la sección 3 deja una fracción alta de outliers y un tópico residual de palabras funcionales. Por eso la sección 5.2 construye un segundo modelo con stopwords ampliadas, parámetros de clustering más cercanos al artículo de referencia y reasignación de outliers. Las visualizaciones de la sección 5.3 usan ese modelo corregido: comparar ambos es parte del aprendizaje, no un apéndice.


In [129]:
# Tabla de tópicos
info = topic_model.get_topic_info()
print(info.head(10))

# Gráfico de barras
fig = topic_model.visualize_barchart(top_n_topics=8)
fig.show()

# Mapa de distancias
fig2 = topic_model.visualize_topics()
fig2.show()

# Documentos representativos
for t in tqdm(info.loc[info['Topic'] != -1, 'Topic'].head(3), desc='Documentos representativos'):
    print(f'\nTópico {t}:')
    for doc in topic_model.get_representative_docs(t)[:2]:
        print('-', doc[:200], '...')


   Topic  Count                                             Name  \
0     -1   5466                              -1_es_ms_ha_poltica   
1      0    514                   0_pensiones_sistema_pensin_afp   
2      1    494  1_educacin_estudiantes_escolar_establecimientos   
3      2    364                                  2_es_ms_nos_ser   
4      3    304   3_constitucional_constitucin_convencin_proceso   
5      4    235             4_inflacin_crecimiento_economa_banco   
6      5    215             5_trabajadores_laboral_trabajo_horas   
7      6    206                6_artculo_derechos_derecho_estado   
8      7    201                   7_salud_isapres_fonasa_sistema   
9      8    180                                   8_es_ms_pas_ha   

                                      Representation  \
0  [es, ms, ha, poltica, ser, si, estado, gobiern...   
1  [pensiones, sistema, pensin, afp, reforma, cot...   
2  [educacin, estudiantes, escolar, establecimien...   
3  [es, ms, nos, ser, todo,

Documentos representativos:   0%|          | 0/3 [00:00<?, ?it/s]


Tópico 0:
- no hay que perder de vista el negocio que está haciendo con nuestra seguridad social eso no desaparece publicada en resumen el de noviembre entrevista a recaredo gálvez investigador fundación sol el a ...
- distribución de la cotización adicional en la reforma de pensiones centro de estudios públicos edición digital n abril puntos de referencia e co n o m í a y p o l í t i c a s p ú b l i c a s centro de ...

Tópico 1:
- clases a distancia debilidades e inequidades del sistema aplicado a básica y media para sortear la pandemia ha pasado un mes desde que tras una tensa reunión en la moneda el ministro de educación raúl ...
- una educación para la vida y su cuidado propuestas para una mejor contribución de la educación chilena a enfrentar la crisis del covid resumen ejecutivo como nunca quizás en nuestra historia reciente  ...

Tópico 2:
- relato acerca de la importancia de la cultura y del arte por enrique matthey artista visual y socio de la casa común hace un par de seman

In [130]:
fechas = pd.to_datetime(df['FechaPublicacion'])
df['Año'] = fechas.dt.year

topics_over_time = topic_model.topics_over_time(
    df['texto_limpio'].tolist(), fechas.tolist(), nr_bins=24
)
fig3 = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=5)
fig3.show()


24it [03:01,  7.56s/it]


### 5.2 Segundo modelo: limpieza de ruido y política de outliers

El modelo de la sección 3 es deliberadamente conservador y deja dos problemas visibles:

1. **Un tópico residual de palabras funcionales** (`es`, `ms`, `nos`, `ser`, `le`, `todo`, `si`, `tambin`), favorecido por la ausencia de tildes en el corpus y por stopwords acentuadas que no calzan con esas formas.
2. **Más del 40 % de outliers**, consecuencia de `min_cluster_size=30`, `min_samples=10` y una proyección de 5 dimensiones. El artículo de referencia usa 10 dimensiones, `min_cluster_size=8`, `min_samples=5` y reasigna los outliers con kNN.

Para separar el efecto de los datos del efecto de los hiperparámetros, construimos un **segundo modelo** con estos cambios:

- Stopwords ampliadas (`STOPWORDS_FIX`): variantes sin tilde, funcionales ausentes y artefactos de scraping. Se conservan los términos políticos sustantivos (gobierno, Estado, país, universidad, constitución, pensiones).
- UMAP con `n_components=10`, `min_dist=0.0`, `metric='cosine'`.
- HDBSCAN con `min_cluster_size=15`, `min_samples=5` y `prediction_data=True`.
- Reasignación de outliers por similitud de embeddings (`strategy='embeddings'`, `threshold=0.1`).

Este segundo modelo no es "el correcto": es un contraste controlado. Si mejora la cobertura y elimina el tópico residual, gana interpretabilidad; si fragmenta en exceso, el modelo conservador era preferible.


#### Stopwords de la segunda pasada

El bloque anterior (sección 2) contiene las stopwords del **modelo base** y la función de limpieza. Este bloque contiene las stopwords del **modelo corregido**: variantes sin tilde, funcionales ausentes, artefactos de scraping y acrónimos de marca. Se mantienen separadas para que cada modelo sea reproducible y para poder comparar el efecto de la limpieza léxica.

In [131]:
# Stopwords de la segunda pasada (modelo corregido, sección 5.2).
# Ampliación para el modelo corregido (sección 5.2): variantes sin tilde,
# funcionales ausentes y artefactos de scraping. No incluye términos de dominio.
STOPWORDS_EXTRA = set("""
abstract ademas adems alguna algunas algunos algún ambas ambos analisis antes apenas aqu aquel
aquella aquellas aquellos aqui asi así aun aunque autor aún bastante cada cl clic click com contacto
contra copyright cual cuales cuando cuanto cuantos cuya cuyo da dan das debe deben debería deberían
demasiado demás desarrollado descarga descargar desde despues dijo dio directorio donde durante edu
ella ellas ello ellos email entonces entre era eran es esa esas ese eso esos esta estaba estaban
estamos estan estar estara estas este esto estos estoy estuvo está están exclusiva expresadas
fellows finalmente fue fueron ha haba haber habia habrá habría había habían hace hacen hacer hacia
han hasta hay hemos hizo incluso integratek investigacion journal le les luego mas me mediante menos
mi miembros mientras mis misma mismas mismo mismos mrdesign ms muy mí necesariamente newsletter ni
ninguna ninguno ningún nos nosotros nuestra nuestras nuestro nuestros opiniones org otra otras otro
otros oxford pdf poca pocas poco pocos podemos podrían porque privacidad publicacion publicada puede
pueden pues qu quien quienes quién quiénes representan reservados routledge s sea sean segn segun
senior ser sera seran seria serian será serán sería serían si sido siendo siguenos sino siquiera slo
sobre solo somos son soy suscribete suscripcion sí sólo tambien tambin también tampoco tan te tena
tenemos tener tenia tenía tenían tiene tienen toda todas todo todos tras traves tu tus tuvo tú usted
ustedes varias varios wiley ya él
""".split())
# Acrónimos de marca detectados por la auditoría de entidades: no aportan tema.
ACRONIMOS_MARCA = {"cep", "lyd", "fpp", "pivotes", "ies", "cdc", "opes", "ical", "ced", "fjg", "algo"}
FUNCIONALES_EXTRA = set("""
actual actuales ahora ahi alla alli apenas aseguro asimismo ayer bastante cosa cosas cuanto cuanta
cuantos cuantas debia dice dicen dijo existe existen existia explico gente hoy igualmente indico
jamas manana mayor menores menos mes meses necesario necesaria necesarios posible posibles principal
principales proxima proximo realiza realizo registro senalo senor senora senores sostuvo suficiente
tal tales yo dia dias ano anos ao aos pas
""".split())
STOPWORDS_FIX = (STOPWORDS | STOPWORDS_EXTRA | ACRONIMOS_MARCA | FUNCIONALES_EXTRA) - {"data", "per", "performance", "approach"}
print(f"Stopwords base: {len(STOPWORDS)} | ampliadas: {len(STOPWORDS_FIX)}")




Stopwords base: 269 | ampliadas: 531


In [132]:
umap_fix = UMAP(n_neighbors=15, n_components=10, min_dist=0.0,
                metric='cosine', random_state=SEED)
hdb_fix = HDBSCAN(min_cluster_size=15, min_samples=5, metric='euclidean',
                  cluster_selection_method='eom', prediction_data=True)
vec_fix = CountVectorizer(stop_words=list(STOPWORDS_FIX), ngram_range=(1, 2),
                          min_df=2)

topic_model_fix = BERTopic(umap_model=umap_fix, hdbscan_model=hdb_fix,
                           vectorizer_model=vec_fix, verbose=True)
topics_fix, probs_fix = topic_model_fix.fit_transform(df['texto_limpio'].tolist(), embeddings)
print('Outliers antes de reasignar:', f"{np.mean(np.array(topics_fix) == -1):.1%}")

topics_fix = topic_model_fix.reduce_outliers(
    df['texto_limpio'].tolist(), topics_fix,
    strategy='embeddings', embeddings=embeddings, threshold=0.1)
topic_model_fix.update_topics(df['texto_limpio'].tolist(), topics=topics_fix,
                              vectorizer_model=vec_fix)

info_fix = topic_model_fix.get_topic_info()
df['topic_fix'] = topics_fix
labels_fix = {int(r.Topic): str(r.Name)[:40] for r in info_fix.itertuples()}
df['topic_label'] = [labels_fix.get(int(t), str(t)) for t in topics_fix]

print('Outliers después de reasignar:', f"{np.mean(np.array(topics_fix) == -1):.1%}")
print(info_fix.head(15).to_string())


2026-11-05 18:32:06,263 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-05 18:32:13,697 - BERTopic - Dimensionality - Completed ✓
2026-11-05 18:32:13,699 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-05 18:32:14,164 - BERTopic - Cluster - Completed ✓
2026-11-05 18:32:14,168 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-05 18:32:25,337 - BERTopic - Representation - Completed ✓
2026-11-05 18:32:30,342 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers antes de reasignar: 41.0%
Outliers después de reasignar: 0.0%
    Topic  Count                                             Name                                                                                                            Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [133]:
from IPython.display import display

info_base = topic_model.get_topic_info()
info_base = info_base[info_base['Topic'] != -1]

comp = pd.DataFrame([
    {'Modelo': 'Base',
     'Outliers %': 100 * np.mean(np.array(topics) == -1),
     'Tópicos': len(info_base),
     'Cobertura %': 100 * (1 - np.mean(np.array(topics) == -1)),
     'Mayor tópico': f"{info_base.iloc[0]['Name'][:38]} ({info_base.iloc[0]['Count'] / len(df):.1%})"},
    {'Modelo': 'Corregido',
     'Outliers %': 100 * np.mean(np.array(topics_fix) == -1),
     'Tópicos': int((info_fix['Topic'] != -1).sum()),
     'Cobertura %': 100 * (1 - np.mean(np.array(topics_fix) == -1)),
     'Mayor tópico': f"{info_fix[info_fix['Topic'] != -1].iloc[0]['Name'][:38]} "
                     f"({info_fix[info_fix['Topic'] != -1].iloc[0]['Count'] / len(df):.1%})"},
])

display(comp.round(1))

fig_comp = px.bar(
    comp.melt(id_vars='Modelo', value_vars=['Outliers %', 'Cobertura %']),
    x='Modelo', y='value', color='variable', barmode='group',
    text_auto='.1f', template='plotly_white',
    title='Outliers y cobertura: modelo base vs. modelo corregido',
)
fig_comp.show()


,Modelo,Outliers %,Tópicos,Cobertura %,Mayor tópico
0,Base,45.8,59,54.2,0_pensiones_sistema_pensin_afp (4.3%)
1,Corregido,0.0,118,100.0,0_educacin_estudiantes_escolar_estable (4.4%)


#### Resultados del modelo corregido

El mismo bloque de inspección de la sección 5.1, pero aplicado a `topic_model_fix`, que usa las stopwords de la segunda pasada. Sirve para comparar etiquetas, tamaños y documentos representativos entre ambos modelos.

In [134]:
info_fix = topic_model_fix.get_topic_info()
print(info_fix.head(10))

# Gráfico de barras
fig = topic_model_fix.visualize_barchart(top_n_topics=8)
fig.show()

# Mapa de distancias
fig2 = topic_model_fix.visualize_topics()
fig2.show()

# Documentos representativos
for t in tqdm(info_fix.loc[info_fix['Topic'] != -1, 'Topic'].head(3),
              desc='Documentos representativos (modelo corregido)'):
    print(f'\nTópico {t}:')
    for doc in topic_model_fix.get_representative_docs(t)[:2]:
        print('-', doc[:200], '...')


   Topic  Count                                             Name  \
0      0    521  0_educacin_estudiantes_escolar_establecimientos   
1      1    510                   1_pensiones_pensin_afp_sistema   
2      2    404                2_litio_ambiental_climtico_minera   
3      3    378   3_constitucional_constitucin_convencin_proceso   
4      4    232             4_elecciones_partido_derecha_eleccin   
5      5    187                   5_salud_isapres_fonasa_sistema   
6      6    218              6_universidad_fundacin_pdc_director   
7      7    240                    7_economa_estado_econmico_est   
8      8    184                      8_mundo_historia_libro_vida   
9      9    146                   9_virus_covid_pandemia_vacunas   

                                      Representation  \
0  [educacin, estudiantes, escolar, establecimien...   
1  [pensiones, pensin, afp, sistema, cotizacin, r...   
2  [litio, ambiental, climtico, minera, cambio cl...   
3  [constitucional, constit

Documentos representativos (modelo corregido):   0%|          | 0/3 [00:00<?, ?it/s]


Tópico 0:
- plan de reactivación educativa y la necesidad de recuperar la asistencia escolar documento disponible luego de anuncios poco concretos esta semana el gobierno informó algunas de las medidas que conten ...
- ley de inclusión escolar mientras la atención del debate en torno a la educación suele estar capturada por el nuevo proyecto del gobierno que pone fin al crédito con aval del estado cae y crea un nuev ...

Tópico 1:
- reforma previsional en chile el gran clamor en medio del estallido social un manifestante besa a una mujer mapuche durante una demostración antigubernamental en santiago chile el de enero de edgard ga ...
- distribución de la cotización adicional en la reforma de pensiones centro de estudios públicos edición digital n abril puntos de referencia e co n o m í a y p o l í t i c a s p ú b l i c a s centro de ...

Tópico 2:
- empresa nacional del litio mientras algunos argumentan que tras el acuerdo entre codelco y sqm la empresa nacional del litio está muerta 

### 5.3 Visualizaciones interactivas

Las cinco figuras siguientes usan el **modelo corregido** y permiten cruzar los tópicos con las variables del corpus. Todas son Plotly: se generan al ejecutar la celda, no son imágenes estáticas.

| Figura | Pregunta | Filtros |
|---|---|---|
| Mapa semántico | ¿Qué organizaciones y temas ocupan qué regiones? | Año (dropdown) y orientación (leyenda) |
| Heatmap tópico × think tank | ¿Quién produce cada agenda? | Año (dropdown) |
| Tópicos en el tiempo | ¿Cómo responde la agenda a las coyunturas? | Orientación (dropdown) y tópico (leyenda) |
| Red de similitud | ¿Qué organizaciones hablan en registros similares? | Umbral de similitud (dropdown) |
| Perfil temático | ¿Quién es generalista y quién especializado? | Métrica (dropdown) |

Los hover muestran título, tópico, año y organización cuando corresponde. Las líneas verticales del gráfico temporal marcan el estallido (2019-10-18), el plebiscito (2020-10-25), el rechazo del primer texto (2022-09-04) y el rechazo del segundo (2023-12-17).


In [135]:
viz_fix = pd.DataFrame({
    'UMAP 1': emb_2d[:, 0],
    'UMAP 2': emb_2d[:, 1],
    'Orientación': df['Orientacion_Politica'].values,
    'Think tank': df['Think_Tank'].values,
    'Año': df['Año'].astype(int).astype(str).values,
    'Tópico': df['topic_label'].values,
    'Título': df['Titulo'].fillna('').str.slice(0, 90).values,
})

colores = {'Derecha': '#d62728', 'Izquierda': '#1f77b4'}
fig_mapa = go.Figure()
for orient in ['Derecha', 'Izquierda']:
    sub = viz_fix[viz_fix['Orientación'] == orient]
    fig_mapa.add_trace(go.Scattergl(
        x=sub['UMAP 1'], y=sub['UMAP 2'], mode='markers', name=orient,
        marker=dict(size=4, opacity=0.55, color=colores[orient]),
        text=sub['Think tank'] + ' · ' + sub['Año'] + '<br>' + sub['Tópico'] + '<br>' + sub['Título'],
        hovertemplate='%{text}<extra></extra>'))

botones = []
for year in ['Todos'] + sorted(viz_fix['Año'].unique()):
    mask = np.ones(len(viz_fix), bool) if year == 'Todos' else (viz_fix['Año'] == year).values
    xs, ys, textos = [], [], []
    for orient in ['Derecha', 'Izquierda']:
        sub = viz_fix[mask & (viz_fix['Orientación'] == orient)]
        xs.append(sub['UMAP 1'].tolist())
        ys.append(sub['UMAP 2'].tolist())
        textos.append((sub['Think tank'] + ' · ' + sub['Año'] + '<br>' + sub['Tópico']).tolist())
    botones.append(dict(label=str(year), method='restyle',
                        args=[{'x': xs, 'y': ys, 'text': textos}]))

fig_mapa.update_layout(
    title='Mapa semántico UMAP 2D (color: orientación; filtro: año)',
    template='plotly_white', height=620,
    updatemenus=[dict(buttons=botones, direction='down', x=1.02, y=1, showactive=True)],
    legend=dict(orientation='h', y=1.08),
)
fig_mapa.show()


In [136]:
top_ids = [int(t) for t in info_fix[info_fix['Topic'] != -1].head(12)['Topic']]
etiquetas = [str(info_fix.loc[info_fix['Topic'] == t, 'Name'].iloc[0])[:36] for t in top_ids]
tts = df['Think_Tank'].value_counts().index.tolist()

def matriz_tt(year):
    d = df if year == 'Todos' else df[df['Año'].astype(int).astype(str) == year]
    total = d.groupby('Think_Tank').size()
    m = pd.crosstab(d['Think_Tank'], d['topic_fix'])
    m = m.reindex(index=tts, columns=top_ids, fill_value=0)
    return m.div(total.reindex(tts).replace(0, np.nan), axis=0).fillna(0)

z0 = matriz_tt('Todos')
fig_heat = go.Figure(go.Heatmap(
    z=z0.values, x=etiquetas, y=tts, colorscale='Blues',
    text=[[f'{v:.1%}' for v in row] for row in z0.values],
    hovertemplate='%{y}<br>%{x}: %{text}<extra></extra>'))

botones_heat = []
for year in ['Todos'] + sorted(df['Año'].astype(int).astype(str).unique()):
    z = matriz_tt(year)
    botones_heat.append(dict(label=str(year), method='restyle',
                             args=[{'z': [z.values],
                                    'text': [[f'{v:.1%}' for v in row] for row in z.values]}]))

fig_heat.update_layout(
    title='Participación de cada tópico dentro de la agenda de cada think tank',
    template='plotly_white', height=720,
    updatemenus=[dict(buttons=botones_heat, direction='down', x=1.02, y=1, showactive=True)],
    xaxis=dict(tickangle=-35),
)
fig_heat.show()


In [137]:
fechas = pd.to_datetime(df['FechaPublicacion'])
top_ts = [int(t) for t in info_fix[info_fix['Topic'] != -1].head(6)['Topic']]
etiquetas_ts = {t: str(info_fix.loc[info_fix['Topic'] == t, 'Name'].iloc[0])[:36] for t in top_ts}

def serie(mask):
    return topic_model_fix.topics_over_time(
        df.loc[mask, 'texto_limpio'].tolist(),
        fechas[mask].tolist(),
        topics=np.asarray(topics_fix)[mask].tolist(),
        nr_bins=24,
    )

tot = {
    'Ambos': serie(np.ones(len(df), bool)),
    'Derecha': serie((df['Orientacion_Politica'] == 'Derecha').values),
    'Izquierda': serie((df['Orientacion_Politica'] == 'Izquierda').values),
}

colores_ts = {t: px.colors.qualitative.Dark24[i] for i, t in enumerate(top_ts)}
fig_ts = go.Figure()
for orient in ['Ambos', 'Derecha', 'Izquierda']:
    for t in top_ts:
        d = tot[orient][tot[orient]['Topic'] == t]
        fig_ts.add_trace(go.Scatter(
            x=d['Timestamp'], y=d['Frequency'], mode='lines',
            name=etiquetas_ts[t], legendgroup=str(t), showlegend=(orient == 'Ambos'),
            line=dict(color=colores_ts[t], width=2),
            visible=(orient == 'Ambos'),
            hovertemplate=f'{orient} · {etiquetas_ts[t]}<br>%{{x}}<br>%{{y}} docs<extra></extra>',
        ))

eventos = {
    '2019-10-18': 'Estallido',
    '2020-10-25': 'Plebiscito',
    '2022-09-04': 'Rechazo 1',
    '2023-12-17': 'Rechazo 2',
}
for fecha, etiqueta in eventos.items():
    fig_ts.add_vline(x=pd.Timestamp(fecha), line_dash='dot', line_color='#555')
    fig_ts.add_annotation(x=pd.Timestamp(fecha), y=1, yref='paper', text=etiqueta,
                          showarrow=False, textangle=-90, font=dict(size=9))

vis = {o: [o == orient for orient in ['Ambos', 'Derecha', 'Izquierda'] for _ in top_ts]
       for o in ['Ambos', 'Derecha', 'Izquierda']}
fig_ts.update_layout(
    title='Evolución temporal de los seis tópicos principales (filtro: orientación)',
    template='plotly_white', height=560,
    updatemenus=[dict(
        buttons=[dict(label=o, method='restyle', args=[{'visible': vis[o]}])
                 for o in ['Ambos', 'Derecha', 'Izquierda']],
        direction='down', x=1.02, y=1, showactive=True)],
    legend=dict(orientation='h', y=-0.2),
)
fig_ts.show()


24it [01:43,  4.33s/it]
24it [01:22,  3.44s/it]
24it [00:45,  1.89s/it]


In [138]:
perfil = pd.crosstab(df['Think_Tank'], df['topic_fix'])
perfil = perfil.drop(columns=[-1], errors='ignore')
perfil = perfil.div(perfil.sum(axis=1), axis=0).fillna(0)

sim = cosine_similarity(perfil.values)
tts_net = perfil.index.tolist()
pos = UMAP(n_neighbors=5, n_components=2, min_dist=0.3, metric='cosine',
           random_state=SEED).fit_transform(perfil.values)
vol = df['Think_Tank'].value_counts().reindex(tts_net).values
orient = [df.loc[df['Think_Tank'] == t, 'Orientacion_Politica'].iloc[0] for t in tts_net]
top3 = {t: ' / '.join(str(info_fix.loc[info_fix['Topic'] == x, 'Name'].iloc[0])[:30]
                      for x in perfil.loc[t].sort_values(ascending=False).head(3).index)
        for t in tts_net}

fig_net = go.Figure()
umbrales = [0.5, 0.7, 0.85, 0.95]
for k, th in enumerate(umbrales):
    ex, ey = [], []
    for i in range(len(tts_net)):
        for j in range(i + 1, len(tts_net)):
            if sim[i, j] >= th:
                ex += [pos[i, 0], pos[j, 0], None]
                ey += [pos[i, 1], pos[j, 1], None]
    fig_net.add_trace(go.Scatter(x=ex, y=ey, mode='lines',
                                 line=dict(width=1, color='#888'),
                                 visible=(k == 0), showlegend=False, hoverinfo='skip'))

fig_net.add_trace(go.Scatter(
    x=pos[:, 0], y=pos[:, 1], mode='markers+text', text=tts_net,
    textposition='top center', textfont=dict(size=9),
    marker=dict(size=8 + 18 * vol / vol.max(),
                color=[colores[o] for o in orient], opacity=0.85),
    customdata=list(zip(vol, orient, [top3[t] for t in tts_net])),
    hovertemplate='%{text}<br>Volumen: %{customdata[0]}<br>Orientación: %{customdata[1]}<br>%{customdata[2]}<extra></extra>',
    showlegend=False))

def visibilidad(k):
    v = [False] * len(umbrales) + [True]
    v[k] = True
    return v

fig_net.update_layout(
    title='Similitud discursiva entre think tanks (perfiles de tópicos)',
    template='plotly_white', height=680,
    updatemenus=[dict(
        buttons=[dict(label=f'≥ {th:.2f}', method='restyle', args=[{'visible': [visibilidad(k)]}])
                 for k, th in enumerate(umbrales)],
        direction='down', x=1.02, y=1, showactive=True)],
)
fig_net.show()


In [139]:
perfil_all = pd.crosstab(df['Think_Tank'], df['topic_fix'])
total = perfil_all.sum(axis=1)
sin_out = perfil_all.drop(columns=[-1], errors='ignore')
share = sin_out.div(total, axis=0).fillna(0)

def entropia(p):
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

metricas_df = pd.DataFrame({
    'Think tank': perfil_all.index,
    'Volumen': total.values,
    'Entropía': [entropia(share.loc[t].values) for t in perfil_all.index],
    'Outliers %': [100 * perfil_all.loc[t].get(-1, 0) / total[t] for t in perfil_all.index],
    'Tópicos (>=5)': [(sin_out.loc[t] >= 5).sum() for t in perfil_all.index],
})
metricas_df['Orientación'] = [
    df.loc[df['Think_Tank'] == t, 'Orientacion_Politica'].iloc[0]
    for t in metricas_df['Think tank']
]
metricas_df = metricas_df.sort_values('Volumen', ascending=False)
opciones = ['Volumen', 'Entropía', 'Outliers %', 'Tópicos (>=5)']

fig_perfil = go.Figure(go.Bar(
    x=metricas_df['Think tank'], y=metricas_df['Volumen'],
    marker_color=[colores[o] for o in metricas_df['Orientación']],
    hovertemplate='%{x}: %{y}<extra></extra>',
))
fig_perfil.update_layout(
    title='Perfil por think tank (selector de métrica)',
    template='plotly_white', height=560,
    updatemenus=[dict(
        buttons=[dict(label=m, method='restyle', args=[{'y': [metricas_df[m].tolist()]}])
                 for m in opciones],
        direction='down', x=1.02, y=1, showactive=True)],
    xaxis=dict(tickangle=-35),
)
fig_perfil.show()


### 5.4 Explorador de tópicos

El explorador combina tres vistas para un tópico seleccionado: sus palabras clave, la serie de tiempo mensual y una segunda serie en **barras apiladas**. La tabla de publicaciones se reemplazó por esa segunda vista para evitar duplicaciones y para comparar composiciones.

- **Leyenda = Think tank**: las barras apiladas muestran la composición del tópico seleccionado por organización (top 8 + Otros).
- **Leyenda = Tópico**: las barras apiladas muestran todos los tópicos filtrados por mes (top 8 + Otros); la línea superior sigue siendo el tópico seleccionado.

El buscador filtra los tópicos por palabra o por nombre. Los filtros de think tank, orientación y tipo restringen la línea y las barras. Las líneas punteadas marcan estallido (2019-10-18), plebiscito (2020-10-25), rechazo 1 (2022-09-04) y rechazo 2 (2023-12-17).

> Si al reejecutar la celda ves una vista anterior duplicada, usa **Restart & Run All**: las salidas de widgets pueden quedar pegadas entre ejecuciones.

In [140]:
import ipywidgets as widgets
from IPython.display import display

catalogo = info_fix[info_fix['Topic'] != -1].copy()
catalogo['Palabras'] = [
    ', '.join(w for w, _ in topic_model_fix.get_topic(int(t)))
    for t in catalogo['Topic']
]
catalogo['Etiqueta'] = (catalogo['Topic'].astype(int).astype(str) + ' · ' +
                        catalogo['Name'].astype(str).str[:60])
catalogo = catalogo.sort_values('Count', ascending=False)
opciones = [(catalogo.loc[i, 'Etiqueta'], int(catalogo.loc[i, 'Topic'])) for i in catalogo.index]

buscador = widgets.Text(placeholder='Buscar tópico o palabra…', description='Buscar:')
selector = widgets.Dropdown(options=opciones, description='Tópico:', layout=widgets.Layout(width='520px'))
filtro_tt = widgets.Dropdown(options=['Todos'] + sorted(df['Think_Tank'].dropna().unique()), value='Todos', description='Think tank:')
filtro_or = widgets.Dropdown(options=['Todos'] + sorted(df['Orientacion_Politica'].dropna().unique()), value='Todos', description='Orientación:')
filtro_tipo = widgets.Dropdown(options=['Todos'] + sorted(df['Tipo_Think_Tank'].dropna().unique()), value='Todos', description='Tipo:')
agrupar = widgets.Dropdown(options=['Think tank', 'Tópico'], value='Think tank', description='Leyenda:')
salida = widgets.Output()

EVENTOS = {
    '2019-10-18': 'Estallido',
    '2020-10-25': 'Plebiscito',
    '2022-09-04': 'Rechazo 1',
    '2023-12-17': 'Rechazo 2',
}


def _filtrar(change):
    q = change['new'].strip().lower()
    if not q:
        nuevas = opciones
    else:
        mask = (catalogo['Etiqueta'].str.lower().str.contains(q, regex=False) |
                catalogo['Palabras'].str.lower().str.contains(q, regex=False))
        nuevas = [(row.Etiqueta, int(row.Topic)) for row in catalogo[mask].itertuples()]
    if nuevas:
        selector.options = nuevas


def _docs_filtrados():
    d = df.copy()
    if filtro_tt.value != 'Todos':
        d = d[d['Think_Tank'] == filtro_tt.value]
    if filtro_or.value != 'Todos':
        d = d[d['Orientacion_Politica'] == filtro_or.value]
    if filtro_tipo.value != 'Todos':
        d = d[d['Tipo_Think_Tank'] == filtro_tipo.value]
    d = d[d['topic_fix'] != -1].copy()
    d['fecha'] = pd.to_datetime(d['FechaPublicacion'], errors='coerce')
    d['mes'] = d['fecha'].dt.to_period('M').dt.to_timestamp()
    return d


def _render(_):
    topic = selector.value
    salida.clear_output(wait=True)
    with salida:
        d_all = _docs_filtrados()
        d = d_all[d_all['topic_fix'] == topic]
        etiqueta = catalogo.loc[catalogo['Topic'] == topic, 'Etiqueta'].iloc[0]
        if d.empty:
            print(etiqueta + ': sin publicaciones con estos filtros.')
            return

        palabras = catalogo.loc[catalogo['Topic'] == topic, 'Palabras'].iloc[0]
        print('Tópico:', etiqueta)
        print('Palabras:', palabras)
        print(f'Documentos del tópico con los filtros activos: {len(d)}')

        serie = d.groupby('mes').size().reset_index(name='docs')
        fig = make_subplots(
            rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
            row_heights=[0.42, 0.58],
            subplot_titles=['Serie de tiempo del tópico',
                            'Composición apilada por ' + agrupar.value.lower()],
        )
        fig.add_trace(go.Scatter(
            x=serie['mes'], y=serie['docs'], mode='lines+markers',
            line=dict(width=2), marker=dict(size=5), name='Total',
            hovertemplate='%{x|%Y-%m}: %{y} documentos<extra></extra>'),
            row=1, col=1)

        base = d if agrupar.value == 'Think tank' else d_all
        col = 'Think_Tank' if agrupar.value == 'Think tank' else 'topic_label'
        conteo = (base.groupby(['mes', col]).size()
                  .reset_index(name='docs'))
        conteo.columns = ['mes', 'grupo', 'docs']
        top = conteo.groupby('grupo')['docs'].sum().nlargest(8).index
        conteo['grupo'] = conteo['grupo'].where(conteo['grupo'].isin(top), 'Otros')
        tabla = conteo.pivot_table(index='mes', columns='grupo', values='docs',
                                   aggfunc='sum', fill_value=0)
        for nombre in tabla.columns:
            fig.add_trace(go.Bar(x=tabla.index, y=tabla[nombre], name=str(nombre)),
                          row=2, col=1)

        for fecha in EVENTOS:
            fig.add_vline(x=pd.Timestamp(fecha), line_dash='dot',
                          line_color='#777', row=1, col=1)
            fig.add_vline(x=pd.Timestamp(fecha), line_dash='dot',
                          line_color='#777', row=2, col=1)

        fig.update_layout(
            barmode='stack', height=720, template='plotly_white',
            title=etiqueta + ' · serie y composición',
            legend=dict(orientation='h', y=-0.18),
            margin=dict(l=50, r=20, t=60, b=80),
        )
        fig.update_yaxes(title_text='Documentos', row=1, col=1)
        fig.update_yaxes(title_text='Documentos', row=2, col=1)
        fig.update_xaxes(title_text='', row=2, col=1)
        display(fig)
        print('Líneas punteadas: estallido, plebiscito y rechazos constitucionales.')


selector.observe(_render, names='value')
for w in (filtro_tt, filtro_or, filtro_tipo, agrupar):
    w.observe(_render, names='value')
buscador.observe(_filtrar, names='value')

display(widgets.VBox([
    widgets.HBox([buscador, selector]),
    widgets.HBox([filtro_tt, filtro_or, filtro_tipo, agrupar]),
    salida,
]))
_render(None)

## 6. Conclusiones

- BERTopic descompone el modelado de tópicos en cuatro decisiones independientes: representación (SBERT), reducción de dimensionalidad (UMAP), agrupamiento (HDBSCAN) y descripción (c-TF-IDF). Cada una tiene supuestos y consecuencias interpretativas.
- La calidad del resultado depende tanto del modelo como del corpus. Un corpus construido por web scraping sobrerrepresenta lo que las organizaciones quisieron publicar y conservar.
- La validación automática es necesaria pero insuficiente. Silhouette, ARI y NLI detectan problemas, pero la interpretación final requiere lectura humana.
- La estabilidad entre semillas y el análisis de top-N palabras permiten distinguir estructura real de artefactos del algoritmo.
- La comparación base/corregido muestra el compromiso entre cobertura y fragmentación: reducir outliers al 0 % fragmentó el modelo en más de 120 tópicos, frente a ~56–60 del modelo base.
- Las stopwords no son un detalle: variantes sin tilde y artefactos de scraping pueden crear un tópico entero de ruido.
- El cruce con metadatos —organización, orientación, tipo y año— convierte una lista de tópicos en un mapa del espacio discursivo. Esa es la contribución analítica del notebook.

### Extensiones posibles

- Modelado dinámico de tópicos (`dynamic_topic_modeling`) para seguir la evolución de un mismo tema.
- Tópicos jerárquicos para agrupar clusters especializados en macro-temas.
- Coherencia de tópicos (c_v) con `gensim` como métrica complementaria.
- Evaluación humana sistemática con una pauta de anotación.
- Reemplazo del modelo de embeddings por uno de mayor dimensión (mpnet) y comparación de resultados.
